## M1 Offline Replay Driver
Quick test: replay one saved M0 `metrics_raw.jsonl` run through `RealCoreEngine` using the M1 scaffold.

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path


def find_phase5_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'plan.md').exists() and (candidate / 'README.md').exists():
            return candidate
        nested = candidate / 'Phase 2' / 'Phase 5'
        if (nested / 'plan.md').exists():
            return nested
    raise FileNotFoundError('Could not locate Phase 5 root from current working directory.')


PHASE5_ROOT = find_phase5_root(Path.cwd())
PHASE2_ROOT = PHASE5_ROOT.parent
PHASE4_ROOT = PHASE2_ROOT / 'Phase 4'

for p in [PHASE5_ROOT, PHASE4_ROOT]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

RUN_TAG = 'pythia2.8b_full'  # swap to: pythia2.8b_full, qwen full, qwen smoke
RESULTS_DIR = PHASE5_ROOT / 'experiments' / 'm0' / 'results' / RUN_TAG
METRICS_PATH = RESULTS_DIR / 'metrics_raw.jsonl'

print('Phase 5 root:', PHASE5_ROOT)
print('Phase 4 root:', PHASE4_ROOT)
print('Results dir: ', RESULTS_DIR)
print('Metrics path:', METRICS_PATH)

Phase 5 root: c:\Users\nscha\Coding\Relationally Embedded Allostatic Learning\Phase 2\Phase 5
Phase 4 root: c:\Users\nscha\Coding\Relationally Embedded Allostatic Learning\Phase 2\Phase 4
Results dir:  c:\Users\nscha\Coding\Relationally Embedded Allostatic Learning\Phase 2\Phase 5\experiments\m0\results\pythia2.8b_full
Metrics path: c:\Users\nscha\Coding\Relationally Embedded Allostatic Learning\Phase 2\Phase 5\experiments\m0\results\pythia2.8b_full\metrics_raw.jsonl


In [2]:
from real_inference import (
    InferenceActionBackend,
    InferenceCoherenceModel,
    InferenceRuntimeState,
    OfflineReplayObservationAdapter,
    load_m0_observations,
)
from real_core.engine import RealCoreEngine

observations = load_m0_observations(METRICS_PATH)
print(f'Loaded {len(observations)} observation snapshots')
print('First observation:', observations[0])

Loaded 24 observation snapshots
First observation: {'token_entropy_mean': 1.8520191397706185, 'token_entropy_std': 1.2085065442801142, 'attention_entropy_mean': 0.5933075328357518, 'hidden_delta_norm_mean': 152.99393024594764, 'tokens_generated': 128.0}


In [3]:
state = InferenceRuntimeState()
observer = OfflineReplayObservationAdapter(observations, runtime_state=state)
actions = InferenceActionBackend(state)
coherence = InferenceCoherenceModel()

engine = RealCoreEngine(
    observer=observer,
    actions=actions,
    coherence=coherence,
    domain_name='phase5_inference_offline_replay',
)

# Quick test pass; increase to observer.max_cycles for deeper replay
cycles = min(20, observer.max_cycles)
summary = engine.run_session(cycles=cycles, consolidate_on_action='rest')

print('Session summary:')
print(' cycles:', summary.cycles)
print(' mean_coherence:', round(summary.mean_coherence, 4))
print(' final_coherence:', round(summary.final_coherence, 4))
print(' gco_counts:', summary.gco_counts)

Session summary:
 cycles: 20
 mean_coherence: 0.6078
 final_coherence: 0.5861
 gco_counts: {'STABLE': 0, 'PARTIAL': 5, 'DEGRADED': 15, 'CRITICAL': 0}


In [4]:
for entry in engine.memory.entries[:5]:
    print({
        'cycle': entry.cycle,
        'action': entry.action,
        'mode': entry.mode,
        'coherence': round(entry.coherence, 4),
        'gco': entry.gco.value,
        'delta': round(entry.delta, 4),
        'state_after_entropy_mean': round(float(entry.state_after.get('token_entropy_mean', 0.0)), 4),
        'state_after_temp': round(float(entry.state_after.get('temperature', 0.0)), 3),
    })

{'cycle': 1, 'action': 'rest', 'mode': 'fluctuation', 'coherence': 0.5712, 'gco': 'DEGRADED', 'delta': 0.0, 'state_after_entropy_mean': 1.7364, 'state_after_temp': 0.8}
{'cycle': 2, 'action': 'observe', 'mode': 'fluctuation', 'coherence': 0.598, 'gco': 'DEGRADED', 'delta': 0.0268, 'state_after_entropy_mean': 1.6943, 'state_after_temp': 0.8}
{'cycle': 3, 'action': 'rest', 'mode': 'fluctuation', 'coherence': 0.6191, 'gco': 'DEGRADED', 'delta': 0.0211, 'state_after_entropy_mean': 1.1235, 'state_after_temp': 0.8}
{'cycle': 4, 'action': 'observe', 'mode': 'constraint', 'coherence': 0.4773, 'gco': 'DEGRADED', 'delta': -0.1418, 'state_after_entropy_mean': 2.1089, 'state_after_temp': 0.8}
{'cycle': 5, 'action': 'adjust_temperature_up', 'mode': 'constraint', 'coherence': 0.4901, 'gco': 'DEGRADED', 'delta': 0.0128, 'state_after_entropy_mean': 1.9125, 'state_after_temp': 0.9}


In [5]:
artifact = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'run_tag': RUN_TAG,
    'metrics_path': str(METRICS_PATH),
    'cycles': summary.cycles,
    'mean_coherence': summary.mean_coherence,
    'final_coherence': summary.final_coherence,
    'gco_counts': summary.gco_counts,
}

out_path = RESULTS_DIR / 'm1_offline_summary.json'
out_path.write_text(json.dumps(artifact, indent=2), encoding='utf-8')
print('Wrote:', out_path)

Wrote: c:\Users\nscha\Coding\Relationally Embedded Allostatic Learning\Phase 2\Phase 5\experiments\m0\results\pythia2.8b_full\m1_offline_summary.json
